In [3]:
import pandas as pd
import numpy as np

In [4]:
new_df = pd.read_csv(r"D:\Desktop\Quora_Duplicate_Project\data\train2.csv")

In [5]:
new_df.dropna(inplace=True)

In [6]:
new_df.drop_duplicates(inplace=True)

In [7]:
import re
import string

CONTRACTIONS = {
    "can't": "cannot",
    "won't": "will not",
    "n't": " not",
    "'re": " are",
    "'s": " is",
    "'d": " would",
    "'ll": " will",
    "'t": " not",
    "'ve": " have",
    "'m": " am",
}


def preprocess_text(text, stem=False):
    """Clean raw text and prepare it for feature extraction.

    Steps:
    - lowercasing
    - normalize contractions
    - remove HTML tags, URLs, emails, mentions, and hashtags
    - remove punctuation, digits, and extra whitespace
    - optional stemming when nltk is available
    """
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    for old, new in CONTRACTIONS.items():
        text = text.replace(old, new)

    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[@#]\w+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()

    if stem:
        try:
            from nltk.stem import PorterStemmer
            stemmer = PorterStemmer()
            tokens = [stemmer.stem(token) for token in tokens]
        except Exception:
            pass

    return " ".join(tokens)


sample_text = "<p>How can I improve my coding skills? Visit https://example.com now! I can't wait :)</p>"
preprocess_text(sample_text)

'how can i improve my coding skills visit now i cannot wait'

In [8]:
new_df['question1'] = new_df['question1'].apply(preprocess_text)
new_df['question2'] = new_df['question2'].apply(preprocess_text)

  Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.0-cp310-cp310-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build

In [9]:
new_df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,0
1,1,3,4,what is the story of kohinoor kohinoor diamond,what would happen if the indian government sto...,0
2,2,5,6,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,0
3,3,7,8,why am i mentally very lonely how can i solve it,find the remainder when math math is divided by,0
4,4,9,10,which one dissolve in water quikly sugar salt ...,which fish would survive in salt water,0


In [10]:
new_df.drop(columns=['id', 'qid1', 'qid2'], axis= 1)

,question1,question2,is_duplicate
0,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,0
1,what is the story of kohinoor kohinoor diamond,what would happen if the indian government sto...,0
2,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,0
3,why am i mentally very lonely how can i solve it,find the remainder when math math is divided by,0
4,which one dissolve in water quikly sugar salt ...,which fish would survive in salt water,0
...,...,...,...
49995,how do you take the derivative of mathfracx math,what is the derivative of mathxyefrac x y math,0
49996,how much space does mac os x yosemite take on ...,can i install mac os x on my hp laptop,0
49997,why are criterium races prearranged and so luc...,can quora helps to solve any current problems ...,0
49998,how can i hack whatsapp account remotely,how can i hack someone is whatsapp account and...,1


In [11]:
all_questions = list(new_df['question1']) + list(new_df['question2'])

In [12]:
#Tokenization
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
tok = Tokenizer(num_words=200000)
tok.fit_on_texts(all_questions)
seq1 = tok.texts_to_sequences(new_df['question1'])
seq2 = tok.texts_to_sequences(new_df['question2'])

In [13]:
padded1 = pad_sequences(seq1, maxlen=25, padding='post')
padded2 = pad_sequences(seq2, maxlen=25, padding='post')

In [17]:
from sklearn.model_selection import train_test_split

y = new_df['is_duplicate'].astype(int).values

X1_train, X1_test, X2_train, X2_test, y_train, y_test = train_test_split(
    padded1,
    padded2,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X1_train:', X1_train.shape)
print('X1_test :', X1_test.shape)
print('X2_train:', X2_train.shape)
print('X2_test :', X2_test.shape)
print('y_train :', y_train.shape)
print('y_test  :', y_test.shape)

X1_train: (40000, 25)
X1_test : (10000, 25)
X2_train: (40000, 25)
X2_test : (10000, 25)
y_train : (40000,)
y_test  : (10000,)


Finding good maxlen

In [18]:
lengths = [len(x) for x in seq1 + seq2]

print(np.percentile(lengths, 90))
print(np.percentile(lengths, 95))
print(np.max(lengths))

18.0
23.0
244


BiLSTM Model

In [21]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM, Dense,
    Dropout, Concatenate, GlobalMaxPool1D, Layer
)
from tensorflow.keras.optimizers import Adam

class AbsDiff(Layer):
    """Element-wise absolute difference between two tensors."""
    def call(self, inputs):
        return tf.abs(inputs[0] - inputs[1])
    def get_config(self):
        return super().get_config()


MAXLEN = 25
VOCAB_SIZE = len(tok.word_index) + 1

input1 = Input(shape=(MAXLEN,))
input2 = Input(shape=(MAXLEN,))

embedding_layer = Embedding(input_dim=VOCAB_SIZE, output_dim=64)

shared_lstm = Bidirectional(
    LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.0)
)

encoded1 = GlobalMaxPool1D()(shared_lstm(embedding_layer(input1)))
encoded2 = GlobalMaxPool1D()(shared_lstm(embedding_layer(input2)))

l1_distance = AbsDiff()([encoded1, encoded2])
merged = Concatenate()([encoded1, encoded2, l1_distance])

x = Dense(128, activation='relu')(merged)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=[input1, input2], outputs=output)
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(
    [X1_train, X2_train], y_train,
    validation_data=([X1_test, X2_test], y_test),
    epochs=3,
    batch_size=128
)


Epoch 1/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 22s 55ms/step - accuracy: 0.6988 - loss: 0.5664 - val_accuracy: 0.7412 - val_loss: 0.5106
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 19s 62ms/step - accuracy: 0.8007 - loss: 0.4251 - val_accuracy: 0.7451 - val_loss: 0.5046
Epoch 3/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 19s 62ms/step - accuracy: 0.8615 - loss: 0.3181 - val_accuracy: 0.7504 - val_loss: 0.5384


In [22]:
loss, accuracy = model.evaluate(
    [X1_test, X2_test],
    y_test
)

print('Accuracy:', accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.7504 - loss: 0.5384
Accuracy: 0.7504000067710876


In [23]:
def predict_duplicate(q1, q2):

    q1 = preprocess_text(q1)
    q2 = preprocess_text(q2)

    q1_seq = tok.texts_to_sequences([q1])
    q2_seq = tok.texts_to_sequences([q2])

    q1_pad = pad_sequences(
        q1_seq,
        maxlen=MAXLEN,
        padding='post'
    )

    q2_pad = pad_sequences(
        q2_seq,
        maxlen=MAXLEN,
        padding='post'
    )

    pred = model.predict([q1_pad, q2_pad])[0][0]

    print('Duplicate Probability:', pred)

    if pred > 0.5:
        print('Duplicate Questions')
    else:
        print('Not Duplicate')

In [24]:
predict_duplicate(
    'How can I learn machine learning?',
    'What is the best way to study ML?'
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 468ms/step
Duplicate Probability: 0.7644659
Duplicate Questions


In [26]:
model.save('BiLSTM.h5')
print('Saved BiLSTM.h5')


Saved BiLSTM.h5


In [ ]:
import pickle

# Save the tokenizer so the Flask app uses the exact same word→index mapping
with open('bilstm_tokenizer.pkl', 'wb') as f:
    pickle.dump(tok, f)

print("Tokenizer saved as bilstm_tokenizer.pkl")

Tokenizer saved as bilstm_tokenizer.pkl


: 